In [2]:
!pip install flask -q
print("✅ Flask installed")

✅ Flask installed


In [3]:
import os
os.makedirs("bot", exist_ok=True)
open("bot/__init__.py", "w").close()
print("✅ bot/ folder created")

✅ bot/ folder created


In [4]:
from flask import Flask, jsonify, request
import threading, time, random

app = Flask(__name__)

@app.route('/fapi/v1/ping', methods=['GET'])
def ping():
    return jsonify({})

@app.route('/fapi/v2/balance', methods=['GET'])
def balance():
    return jsonify([
        {"asset": "USDT", "balance": "10000.00", "availableBalance": "10000.00"},
        {"asset": "BNB",  "balance": "100.00",   "availableBalance": "100.00"}
    ])

@app.route('/fapi/v1/order', methods=['POST'])
def place_order():
    data = request.values
    return jsonify({
        "orderId":     random.randint(100000, 999999),
        "symbol":      data.get("symbol", "BTCUSDT"),
        "side":        data.get("side", "BUY"),
        "type":        data.get("type", "MARKET"),
        "origQty":     data.get("quantity", "0.01"),
        "executedQty": data.get("quantity", "0.01"),
        "avgPrice":    "67432.10",
        "status":      "FILLED",
        "timeInForce": data.get("timeInForce", "GTC")
    })

t = threading.Thread(
    target=lambda: app.run(port=8000, debug=False, use_reloader=False)
)
t.daemon = True
t.start()
time.sleep(2)
print("✅ Mock Binance server running at http://localhost:8000")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8000
INFO:werkzeug:Press CTRL+C to quit


✅ Mock Binance server running at http://localhost:8000


In [5]:
%%writefile bot/client.py
"""
client.py — Binance Futures client wrapper
Uses direct REST calls against mock server (or real testnet).
Change BASE_URL to switch between mock and real testnet.
"""
import hmac
import hashlib
import time
import logging
import requests
from urllib.parse import urlencode

BASE_URL = "http://localhost:8000"   # swap to testnet URL when available
logger = logging.getLogger(__name__)


class BinanceClient:

    def __init__(self, api_key: str, api_secret: str):
        self.api_key    = api_key
        self.api_secret = api_secret
        self.session    = requests.Session()
        self.session.headers.update({"X-MBX-APIKEY": self.api_key})
        logger.info("BinanceClient initialised")

    def _sign(self, params: dict) -> dict:
        """Add timestamp + HMAC-SHA256 signature to params."""
        params["timestamp"] = int(time.time() * 1000)
        query_string = urlencode(params)
        signature = hmac.new(
            self.api_secret.encode("utf-8"),
            query_string.encode("utf-8"),
            hashlib.sha256
        ).hexdigest()
        params["signature"] = signature
        return params

    def ping(self) -> bool:
        """Check connectivity."""
        try:
            r = self.session.get(f"{BASE_URL}/fapi/v1/ping", timeout=10)
            r.raise_for_status()
            logger.info("Ping successful")
            return True
        except Exception as e:
            logger.error(f"Ping failed: {e}")
            return False

    def get_balance(self) -> list:
        """Fetch non-zero account balances."""
        params = self._sign({})
        r = self.session.get(
            f"{BASE_URL}/fapi/v2/balance",
            params=params,
            timeout=10
        )
        r.raise_for_status()
        balances = r.json()
        non_zero = [b for b in balances if float(b["balance"]) > 0]
        logger.info(f"Balance fetched — {len(non_zero)} asset(s) found")
        return non_zero

    def get_raw_session(self):
        """Expose session and _sign for orders.py."""
        return self.session, self._sign

Writing bot/client.py


In [6]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.FileHandler("trading_bot.log"),
        logging.StreamHandler()
    ]
)
print("✅ Logging ready — writing to trading_bot.log")

✅ Logging ready — writing to trading_bot.log


In [7]:
import sys

# Clear cached imports
for mod in list(sys.modules.keys()):
    if mod.startswith("bot"):
        del sys.modules[mod]

from bot.client import BinanceClient

# Use any dummy keys with mock server
client = BinanceClient(
    api_key="test_api_key_123",
    api_secret="test_api_secret_456"
)

# Test 1: Ping
ok = client.ping()
print(f"Ping:  {'✅ Connected' if ok else '❌ Failed'}")

# Test 2: Balance
print("\n💰 Account Balances:")
balances = client.get_balance()
for b in balances:
    print(f"   {b['asset']:8s}  →  {float(b['balance']):>12,.2f}")

INFO:werkzeug:127.0.0.1 - - [20/May/2026 10:15:21] "GET /fapi/v1/ping HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/May/2026 10:15:21] "GET /fapi/v2/balance?timestamp=1779272121914&signature=7c423cf9a770de9988c14f80ea3a12d3f0b0ac887342cd29191235b744ef862c HTTP/1.1" 200 -


Ping:  ✅ Connected

💰 Account Balances:
   USDT      →     10,000.00
   BNB       →        100.00


In [8]:
%%writefile bot/validators.py
"""
validators.py — Input validation before any order is sent.
Catches bad inputs early so we never send garbage to the API.
"""

VALID_SIDES = {"BUY", "SELL"}
VALID_ORDER_TYPES = {"MARKET", "LIMIT"}


def validate_order(symbol: str, side: str, order_type: str,
                   quantity: float, price: float = None) -> None:
    """
    Validate all order inputs.
    Raises ValueError with a clear message if anything is wrong.
    """

    # Symbol
    if not symbol or not isinstance(symbol, str):
        raise ValueError("Symbol must be a non-empty string e.g. BTCUSDT")
    if not symbol.endswith("USDT"):
        raise ValueError(f"Symbol '{symbol}' must end with USDT e.g. BTCUSDT")

    # Side
    if side.upper() not in VALID_SIDES:
        raise ValueError(f"Side must be BUY or SELL, got '{side}'")

    # Order type
    if order_type.upper() not in VALID_ORDER_TYPES:
        raise ValueError(f"Order type must be MARKET or LIMIT, got '{order_type}'")

    # Quantity
    if not isinstance(quantity, (int, float)) or quantity <= 0:
        raise ValueError(f"Quantity must be a positive number, got '{quantity}'")

    # Price — only required for LIMIT orders
    if order_type.upper() == "LIMIT":
        if price is None:
            raise ValueError("Price is required for LIMIT orders")
        if not isinstance(price, (int, float)) or price <= 0:
            raise ValueError(f"Price must be a positive number, got '{price}'")

    if order_type.upper() == "MARKET" and price is not None:
        raise ValueError("Price should not be set for MARKET orders")

Writing bot/validators.py


In [9]:
%%writefile bot/orders.py
"""
orders.py — Order placement logic.
Talks to the API, returns clean structured result.
"""
import logging
from urllib.parse import urlencode
from bot.validators import validate_order

BASE_URL = "http://localhost:8000"   # same as client.py
logger = logging.getLogger(__name__)


def place_order(session, sign_fn, symbol: str, side: str,
                order_type: str, quantity: float, price: float = None) -> dict:
    """
    Place a MARKET or LIMIT order.
    Returns a clean result dict with order details.
    """

    # Step 1 — Validate inputs first
    symbol     = symbol.upper()
    side       = side.upper()
    order_type = order_type.upper()
    validate_order(symbol, side, order_type, quantity, price)

    # Step 2 — Build params
    params = {
        "symbol":   symbol,
        "side":     side,
        "type":     order_type,
        "quantity": quantity,
    }
    if order_type == "LIMIT":
        params["price"]       = price
        params["timeInForce"] = "GTC"   # Good Till Cancelled

    # Step 3 — Log the request
    logger.info(f"Placing order → {params}")
    print(f"\n📤 Order Request:")
    print(f"   Symbol   : {symbol}")
    print(f"   Side     : {side}")
    print(f"   Type     : {order_type}")
    print(f"   Quantity : {quantity}")
    if price:
        print(f"   Price    : {price}")

    # Step 4 — Sign and send
    signed_params  = sign_fn(params)
    query_string   = urlencode(signed_params)
    r = session.post(
        f"{BASE_URL}/fapi/v1/order",
        data=signed_params,
        timeout=10
    )
    r.raise_for_status()
    response = r.json()

    # Step 5 — Log the response
    logger.info(f"Order response → {response}")

    # Step 6 — Return clean result
    result = {
        "orderId":     response.get("orderId"),
        "symbol":      response.get("symbol"),
        "side":        response.get("side"),
        "type":        response.get("type"),
        "status":      response.get("status"),
        "executedQty": response.get("executedQty"),
        "avgPrice":    response.get("avgPrice"),
    }
    return result


def print_order_result(result: dict, success: bool = True) -> None:
    """Print a clean order result summary."""
    if success:
        print(f"\n✅ Order Successful!")
    else:
        print(f"\n❌ Order Failed!")
    print(f"   Order ID     : {result.get('orderId')}")
    print(f"   Symbol       : {result.get('symbol')}")
    print(f"   Side         : {result.get('side')}")
    print(f"   Type         : {result.get('type')}")
    print(f"   Status       : {result.get('status')}")
    print(f"   Executed Qty : {result.get('executedQty')}")
    print(f"   Avg Price    : {result.get('avgPrice')}")

Writing bot/orders.py


In [10]:
import sys, logging

# Clear cached imports
for mod in list(sys.modules.keys()):
    if mod.startswith("bot"):
        del sys.modules[mod]

from bot.client import BinanceClient
from bot.orders import place_order, print_order_result

client = BinanceClient("test_key_123", "test_secret_456")
session, sign_fn = client.get_raw_session()

print("=" * 45)
print("        TEST 1 — MARKET BUY ORDER")
print("=" * 45)

try:
    result = place_order(
        session    = session,
        sign_fn    = sign_fn,
        symbol     = "BTCUSDT",
        side       = "BUY",
        order_type = "MARKET",
        quantity   = 0.01
    )
    print_order_result(result, success=True)
except Exception as e:
    print(f"❌ Error: {e}")

INFO:werkzeug:127.0.0.1 - - [20/May/2026 10:16:00] "POST /fapi/v1/order HTTP/1.1" 200 -


        TEST 1 — MARKET BUY ORDER

📤 Order Request:
   Symbol   : BTCUSDT
   Side     : BUY
   Type     : MARKET
   Quantity : 0.01

✅ Order Successful!
   Order ID     : 786619
   Symbol       : BTCUSDT
   Side         : BUY
   Type         : MARKET
   Status       : FILLED
   Executed Qty : 0.01
   Avg Price    : 67432.10


In [11]:
print("=" * 45)
print("        TEST 2 — LIMIT SELL ORDER")
print("=" * 45)

try:
    result = place_order(
        session    = session,
        sign_fn    = sign_fn,
        symbol     = "BTCUSDT",
        side       = "SELL",
        order_type = "LIMIT",
        quantity   = 0.01,
        price      = 70000.00
    )
    print_order_result(result, success=True)
except Exception as e:
    print(f"❌ Error: {e}")

INFO:werkzeug:127.0.0.1 - - [20/May/2026 10:16:29] "POST /fapi/v1/order HTTP/1.1" 200 -


        TEST 2 — LIMIT SELL ORDER

📤 Order Request:
   Symbol   : BTCUSDT
   Side     : SELL
   Type     : LIMIT
   Quantity : 0.01
   Price    : 70000.0

✅ Order Successful!
   Order ID     : 699687
   Symbol       : BTCUSDT
   Side         : SELL
   Type         : LIMIT
   Status       : FILLED
   Executed Qty : 0.01
   Avg Price    : 67432.10


In [12]:
print("=" * 45)
print("   TEST 3 — VALIDATION ERROR CHECKS")
print("=" * 45)

tests = [
    # (description, kwargs)
    ("LIMIT order missing price",
     dict(symbol="BTCUSDT", side="BUY", order_type="LIMIT", quantity=0.01)),

    ("Invalid side",
     dict(symbol="BTCUSDT", side="HOLD", order_type="MARKET", quantity=0.01)),

    ("Negative quantity",
     dict(symbol="BTCUSDT", side="BUY", order_type="MARKET", quantity=-5)),

    ("Wrong symbol format",
     dict(symbol="BTCETH", side="BUY", order_type="MARKET", quantity=0.01)),
]

for desc, kwargs in tests:
    try:
        place_order(session, sign_fn, **kwargs)
        print(f"  ⚠️  {desc}: should have failed!")
    except ValueError as e:
        print(f"  ✅ {desc}: caught → {e}")

   TEST 3 — VALIDATION ERROR CHECKS
  ✅ LIMIT order missing price: caught → Price is required for LIMIT orders
  ✅ Invalid side: caught → Side must be BUY or SELL, got 'HOLD'
  ✅ Negative quantity: caught → Quantity must be a positive number, got '-5'
  ✅ Wrong symbol format: caught → Symbol 'BTCETH' must end with USDT e.g. BTCUSDT


In [13]:
%%writefile bot/logging_config.py
"""
logging_config.py — Centralized logging setup.
Logs everything to file AND prints to console.
"""
import logging
import os


def setup_logging(log_file: str = "trading_bot.log") -> logging.Logger:
    """
    Configure logging for the entire bot.
    - INFO and above → console
    - DEBUG and above → log file
    Returns the root logger.
    """

    # Create logs folder if needed
    log_dir = os.path.dirname(log_file)
    if log_dir:
        os.makedirs(log_dir, exist_ok=True)

    # Root logger
    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)

    # Formatter
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)-20s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    # Handler 1 — File (DEBUG level — captures everything)
    file_handler = logging.FileHandler(log_file, mode="a", encoding="utf-8")
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    # Handler 2 — Console (INFO level — only important stuff)
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    # Avoid duplicate handlers if called multiple times
    if not logger.handlers:
        logger.addHandler(file_handler)
        logger.addHandler(console_handler)
    else:
        logger.handlers.clear()
        logger.addHandler(file_handler)
        logger.addHandler(console_handler)

    return logger

Writing bot/logging_config.py


In [14]:
%%writefile bot/orders.py
"""
orders.py — Order placement with full error handling and logging.
"""
import logging
import requests
from urllib.parse import urlencode
from bot.validators import validate_order

BASE_URL = "http://localhost:8000"
logger = logging.getLogger(__name__)


def place_order(session, sign_fn, symbol: str, side: str,
                order_type: str, quantity: float, price: float = None) -> dict:
    """
    Place a MARKET or LIMIT order with full error handling.
    Returns a clean result dict.
    Raises ValueError for bad inputs, RuntimeError for API failures.
    """

    # Step 1 — Sanitize inputs
    symbol     = symbol.strip().upper()
    side       = side.strip().upper()
    order_type = order_type.strip().upper()

    # Step 2 — Validate
    try:
        validate_order(symbol, side, order_type, quantity, price)
    except ValueError as e:
        logger.warning(f"Validation failed: {e}")
        raise

    # Step 3 — Build params
    params = {
        "symbol":   symbol,
        "side":     side,
        "type":     order_type,
        "quantity": quantity,
    }
    if order_type == "LIMIT":
        params["price"]       = price
        params["timeInForce"] = "GTC"

    # Step 4 — Log request
    logger.info(f"ORDER REQUEST | {symbol} | {side} | {order_type} "
                f"| qty={quantity}" + (f" | price={price}" if price else ""))

    # Step 5 — Print request summary
    print(f"\n{'='*48}")
    print(f"  📤 ORDER REQUEST SUMMARY")
    print(f"{'='*48}")
    print(f"  Symbol     : {symbol}")
    print(f"  Side       : {side}")
    print(f"  Type       : {order_type}")
    print(f"  Quantity   : {quantity}")
    if price:
        print(f"  Price      : {price}")
    print(f"{'='*48}")

    # Step 6 — Sign and send
    try:
        signed_params = sign_fn(params)
        r = session.post(
            f"{BASE_URL}/fapi/v1/order",
            data=signed_params,
            timeout=10
        )
        r.raise_for_status()

    except requests.exceptions.Timeout:
        msg = "Request timed out — Binance API not responding"
        logger.error(msg)
        raise RuntimeError(msg)

    except requests.exceptions.ConnectionError:
        msg = "Connection error — check your internet connection"
        logger.error(msg)
        raise RuntimeError(msg)

    except requests.exceptions.HTTPError as e:
        msg = f"API HTTP error: {e} | Response: {r.text}"
        logger.error(msg)
        raise RuntimeError(msg)

    # Step 7 — Parse response
    try:
        response = r.json()
    except Exception:
        msg = f"Could not parse API response: {r.text}"
        logger.error(msg)
        raise RuntimeError(msg)

    # Step 8 — Log response
    logger.info(f"ORDER RESPONSE | orderId={response.get('orderId')} "
                f"| status={response.get('status')} "
                f"| executedQty={response.get('executedQty')} "
                f"| avgPrice={response.get('avgPrice')}")

    # Step 9 — Build clean result
    result = {
        "orderId":     response.get("orderId"),
        "symbol":      response.get("symbol"),
        "side":        response.get("side"),
        "type":        response.get("type"),
        "status":      response.get("status"),
        "executedQty": response.get("executedQty"),
        "avgPrice":    response.get("avgPrice"),
    }

    return result


def print_order_result(result: dict, success: bool = True) -> None:
    """Print a clean formatted order result."""
    print(f"\n{'='*48}")
    print(f"  {'✅ ORDER SUCCESS' if success else '❌ ORDER FAILED'}")
    print(f"{'='*48}")
    print(f"  Order ID     : {result.get('orderId')}")
    print(f"  Symbol       : {result.get('symbol')}")
    print(f"  Side         : {result.get('side')}")
    print(f"  Type         : {result.get('type')}")
    print(f"  Status       : {result.get('status')}")
    print(f"  Executed Qty : {result.get('executedQty')}")
    print(f"  Avg Price    : {result.get('avgPrice')}")
    print(f"{'='*48}")

Overwriting bot/orders.py


In [15]:
import sys

# Clear cache
for mod in list(sys.modules.keys()):
    if mod.startswith("bot"):
        del sys.modules[mod]

from bot.logging_config import setup_logging
logger = setup_logging("trading_bot.log")

logger.debug("This is a DEBUG message — file only")
logger.info("This is an INFO message — file + console")
logger.warning("This is a WARNING — file + console")
logger.error("This is an ERROR — file + console")

print("\n✅ Logging working — check trading_bot.log in the files panel")

2026-05-20 10:28:00 | INFO     | root                 | This is an INFO message — file + console
2026-05-20 10:28:00 | WARNING  | root                 | This is a WARNING — file + console
2026-05-20 10:28:00 | ERROR    | root                 | This is an ERROR — file + console



✅ Logging working — check trading_bot.log in the files panel


In [16]:
from bot.client import BinanceClient
from bot.orders import place_order, print_order_result

client = BinanceClient("test_key_123", "test_secret_456")
session, sign_fn = client.get_raw_session()

print("TEST 1 — MARKET BUY")
try:
    result = place_order(
        session    = session,
        sign_fn    = sign_fn,
        symbol     = "BTCUSDT",
        side       = "BUY",
        order_type = "MARKET",
        quantity   = 0.01
    )
    print_order_result(result, success=True)
except Exception as e:
    print(f"❌ {e}")

2026-05-20 10:28:31 | INFO     | bot.client           | BinanceClient initialised
2026-05-20 10:28:31 | INFO     | bot.orders           | ORDER REQUEST | BTCUSDT | BUY | MARKET | qty=0.01
2026-05-20 10:28:31 | INFO     | werkzeug             | 127.0.0.1 - - [20/May/2026 10:28:31] "POST /fapi/v1/order HTTP/1.1" 200 -
2026-05-20 10:28:31 | INFO     | bot.orders           | ORDER RESPONSE | orderId=940216 | status=FILLED | executedQty=0.01 | avgPrice=67432.10


TEST 1 — MARKET BUY

  📤 ORDER REQUEST SUMMARY
  Symbol     : BTCUSDT
  Side       : BUY
  Type       : MARKET
  Quantity   : 0.01

  ✅ ORDER SUCCESS
  Order ID     : 940216
  Symbol       : BTCUSDT
  Side         : BUY
  Type         : MARKET
  Status       : FILLED
  Executed Qty : 0.01
  Avg Price    : 67432.10


In [17]:
print("TEST 2 — LIMIT SELL")
try:
    result = place_order(
        session    = session,
        sign_fn    = sign_fn,
        symbol     = "BTCUSDT",
        side       = "SELL",
        order_type = "LIMIT",
        quantity   = 0.01,
        price      = 70000.00
    )
    print_order_result(result, success=True)
except Exception as e:
    print(f"❌ {e}")

2026-05-20 10:28:43 | INFO     | bot.orders           | ORDER REQUEST | BTCUSDT | SELL | LIMIT | qty=0.01 | price=70000.0
2026-05-20 10:28:43 | INFO     | werkzeug             | 127.0.0.1 - - [20/May/2026 10:28:43] "POST /fapi/v1/order HTTP/1.1" 200 -
2026-05-20 10:28:43 | INFO     | bot.orders           | ORDER RESPONSE | orderId=921091 | status=FILLED | executedQty=0.01 | avgPrice=67432.10


TEST 2 — LIMIT SELL

  📤 ORDER REQUEST SUMMARY
  Symbol     : BTCUSDT
  Side       : SELL
  Type       : LIMIT
  Quantity   : 0.01
  Price      : 70000.0

  ✅ ORDER SUCCESS
  Order ID     : 921091
  Symbol       : BTCUSDT
  Side         : SELL
  Type         : LIMIT
  Status       : FILLED
  Executed Qty : 0.01
  Avg Price    : 67432.10


In [18]:
print("📋 Contents of trading_bot.log:\n")
with open("trading_bot.log", "r") as f:
    print(f.read())

📋 Contents of trading_bot.log:

2026-05-20 10:28:00 | DEBUG    | root                 | This is a DEBUG message — file only
2026-05-20 10:28:00 | INFO     | root                 | This is an INFO message — file + console
2026-05-20 10:28:00 | WARNING  | root                 | This is a WARNING — file + console
2026-05-20 10:28:00 | ERROR    | root                 | This is an ERROR — file + console
2026-05-20 10:28:31 | INFO     | bot.client           | BinanceClient initialised
2026-05-20 10:28:31 | INFO     | bot.orders           | ORDER REQUEST | BTCUSDT | BUY | MARKET | qty=0.01
2026-05-20 10:28:31 | DEBUG    | urllib3.connectionpool | Starting new HTTP connection (1): localhost:8000
2026-05-20 10:28:31 | INFO     | werkzeug             | 127.0.0.1 - - [20/May/2026 10:28:31] "POST /fapi/v1/order HTTP/1.1" 200 -
2026-05-20 10:28:31 | DEBUG    | urllib3.connectionpool | http://localhost:8000 "POST /fapi/v1/order HTTP/1.1" 200 165
2026-05-20 10:28:31 | INFO     | bot.orders           

In [19]:
%%writefile cli.py
"""
cli.py — Command Line Interface entry point.
Accepts user arguments and calls the order placement logic.

Usage examples:
  python cli.py --symbol BTCUSDT --side BUY --type MARKET --qty 0.01
  python cli.py --symbol BTCUSDT --side SELL --type LIMIT --qty 0.01 --price 70000
"""
import argparse
import sys
import logging
from bot.logging_config import setup_logging
from bot.client import BinanceClient
from bot.orders import place_order, print_order_result

# ── Hardcoded for mock server demo
# Replace with userdata.get() or os.environ when using real testnet
API_KEY    = "test_key_123"
API_SECRET = "test_secret_456"


def build_parser() -> argparse.ArgumentParser:
    """Build and return the CLI argument parser."""
    parser = argparse.ArgumentParser(
        prog="trading_bot",
        description="Binance Futures Testnet Trading Bot (USDT-M)",
        formatter_class=argparse.RawTextHelpFormatter,
        epilog=(
            "Examples:\n"
            "  python cli.py --symbol BTCUSDT --side BUY  --type MARKET --qty 0.01\n"
            "  python cli.py --symbol ETHUSDT --side SELL --type LIMIT  --qty 0.1 --price 3500\n"
        )
    )
    parser.add_argument(
        "--symbol", required=True,
        help="Trading pair e.g. BTCUSDT, ETHUSDT"
    )
    parser.add_argument(
        "--side", required=True, choices=["BUY", "SELL"],
        help="Order side: BUY or SELL"
    )
    parser.add_argument(
        "--type", required=True, dest="order_type",
        choices=["MARKET", "LIMIT"],
        help="Order type: MARKET or LIMIT"
    )
    parser.add_argument(
        "--qty", required=True, type=float,
        help="Order quantity e.g. 0.01"
    )
    parser.add_argument(
        "--price", required=False, type=float, default=None,
        help="Limit price (required for LIMIT orders)"
    )
    return parser


def main():
    """Main entry point — parse args, validate, place order."""
    setup_logging("trading_bot.log")
    logger = logging.getLogger(__name__)

    parser = build_parser()
    args   = parser.parse_args()

    logger.info(f"CLI called | symbol={args.symbol} side={args.side} "
                f"type={args.order_type} qty={args.qty} price={args.price}")

    # Validate LIMIT price early with helpful message
    if args.order_type == "LIMIT" and args.price is None:
        parser.error("--price is required when --type is LIMIT")

    # Connect
    try:
        client           = BinanceClient(API_KEY, API_SECRET)
        session, sign_fn = client.get_raw_session()
    except Exception as e:
        logger.error(f"Failed to initialise client: {e}")
        print(f"❌ Could not connect: {e}")
        sys.exit(1)

    # Place order
    try:
        result = place_order(
            session    = session,
            sign_fn    = sign_fn,
            symbol     = args.symbol,
            side       = args.side,
            order_type = args.order_type,
            quantity   = args.qty,
            price      = args.price
        )
        print_order_result(result, success=True)
        logger.info("Order placed successfully")
        sys.exit(0)

    except ValueError as e:
        logger.warning(f"Invalid input: {e}")
        print(f"\n❌ Invalid input: {e}")
        sys.exit(1)

    except RuntimeError as e:
        logger.error(f"Order failed: {e}")
        print(f"\n❌ Order failed: {e}")
        sys.exit(1)

    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        print(f"\n❌ Unexpected error: {e}")
        sys.exit(1)


if __name__ == "__main__":
    main()

Writing cli.py


In [20]:
!python cli.py --symbol BTCUSDT --side BUY --type MARKET --qty 0.01

2026-05-20 10:30:31 | INFO     | werkzeug             | 127.0.0.1 - - [20/May/2026 10:30:31] "POST /fapi/v1/order HTTP/1.1" 200 -


2026-05-20 10:30:31 | INFO     | __main__             | CLI called | symbol=BTCUSDT side=BUY type=MARKET qty=0.01 price=None
2026-05-20 10:30:31 | INFO     | bot.client           | BinanceClient initialised
2026-05-20 10:30:31 | INFO     | bot.orders           | ORDER REQUEST | BTCUSDT | BUY | MARKET | qty=0.01

  📤 ORDER REQUEST SUMMARY
  Symbol     : BTCUSDT
  Side       : BUY
  Type       : MARKET
  Quantity   : 0.01
2026-05-20 10:30:31 | INFO     | bot.orders           | ORDER RESPONSE | orderId=238033 | status=FILLED | executedQty=0.01 | avgPrice=67432.10

  ✅ ORDER SUCCESS
  Order ID     : 238033
  Symbol       : BTCUSDT
  Side         : BUY
  Type         : MARKET
  Status       : FILLED
  Executed Qty : 0.01
  Avg Price    : 67432.10
2026-05-20 10:30:31 | INFO     | __main__             | Order placed successfully


In [21]:
!python cli.py --symbol BTCUSDT --side SELL --type LIMIT --qty 0.01 --price 70000

2026-05-20 10:30:47 | INFO     | werkzeug             | 127.0.0.1 - - [20/May/2026 10:30:47] "POST /fapi/v1/order HTTP/1.1" 200 -


2026-05-20 10:30:47 | INFO     | __main__             | CLI called | symbol=BTCUSDT side=SELL type=LIMIT qty=0.01 price=70000.0
2026-05-20 10:30:47 | INFO     | bot.client           | BinanceClient initialised
2026-05-20 10:30:47 | INFO     | bot.orders           | ORDER REQUEST | BTCUSDT | SELL | LIMIT | qty=0.01 | price=70000.0

  📤 ORDER REQUEST SUMMARY
  Symbol     : BTCUSDT
  Side       : SELL
  Type       : LIMIT
  Quantity   : 0.01
  Price      : 70000.0
2026-05-20 10:30:47 | INFO     | bot.orders           | ORDER RESPONSE | orderId=733613 | status=FILLED | executedQty=0.01 | avgPrice=67432.10

  ✅ ORDER SUCCESS
  Order ID     : 733613
  Symbol       : BTCUSDT
  Side         : SELL
  Type         : LIMIT
  Status       : FILLED
  Executed Qty : 0.01
  Avg Price    : 67432.10
2026-05-20 10:30:47 | INFO     | __main__             | Order placed successfully


In [22]:
# Test 1 — LIMIT without price
print("TEST: LIMIT without price")
!python cli.py --symbol BTCUSDT --side BUY --type LIMIT --qty 0.01

# Test 2 — Invalid qty
print("\nTEST: Negative quantity")
!python cli.py --symbol BTCUSDT --side BUY --type MARKET --qty -1

TEST: LIMIT without price
2026-05-20 10:30:59 | INFO     | __main__             | CLI called | symbol=BTCUSDT side=BUY type=LIMIT qty=0.01 price=None
usage: trading_bot [-h] --symbol SYMBOL --side {BUY,SELL} --type
                   {MARKET,LIMIT} --qty QTY [--price PRICE]
trading_bot: error: --price is required when --type is LIMIT

TEST: Negative quantity
2026-05-20 10:30:59 | INFO     | __main__             | CLI called | symbol=BTCUSDT side=BUY type=MARKET qty=-1.0 price=None
2026-05-20 10:30:59 | INFO     | bot.client           | BinanceClient initialised
2026-05-20 10:30:59 | WARNING  | bot.orders           | Validation failed: Quantity must be a positive number, got '-1.0'
2026-05-20 10:30:59 | WARNING  | __main__             | Invalid input: Quantity must be a positive number, got '-1.0'

❌ Invalid input: Quantity must be a positive number, got '-1.0'


In [23]:
%%writefile requirements.txt
requests>=2.31.0
flask>=3.0.0

Writing requirements.txt


In [24]:
print("📋 Final trading_bot.log contents:\n")
with open("trading_bot.log", "r") as f:
    print(f.read())

📋 Final trading_bot.log contents:

2026-05-20 10:28:00 | DEBUG    | root                 | This is a DEBUG message — file only
2026-05-20 10:28:00 | INFO     | root                 | This is an INFO message — file + console
2026-05-20 10:28:00 | WARNING  | root                 | This is a WARNING — file + console
2026-05-20 10:28:00 | ERROR    | root                 | This is an ERROR — file + console
2026-05-20 10:28:31 | INFO     | bot.client           | BinanceClient initialised
2026-05-20 10:28:31 | INFO     | bot.orders           | ORDER REQUEST | BTCUSDT | BUY | MARKET | qty=0.01
2026-05-20 10:28:31 | DEBUG    | urllib3.connectionpool | Starting new HTTP connection (1): localhost:8000
2026-05-20 10:28:31 | INFO     | werkzeug             | 127.0.0.1 - - [20/May/2026 10:28:31] "POST /fapi/v1/order HTTP/1.1" 200 -
2026-05-20 10:28:31 | DEBUG    | urllib3.connectionpool | http://localhost:8000 "POST /fapi/v1/order HTTP/1.1" 200 165
2026-05-20 10:28:31 | INFO     | bot.orders        

In [27]:
from google.colab import files

download_files = [
    "bot/__init__.py",
    "bot/client.py",
    "bot/orders.py",
    "bot/validators.py",
    "bot/logging_config.py",
    "cli.py",
    "requirements.txt",
    "trading_bot.log",
]

for f in download_files:
    import os
    if os.path.exists(f):
        files.download(f)
        print(f"✅ Downloaded: {f}")
    else:
        print(f"⚠️  Missing: {f}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: bot/__init__.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✅ Downloaded: requirements.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: trading_bot.log
